# SD3.5 inpaint-EDIT — random skeleton placement

Adds people to background images at **random but constrained** positions: the
system samples a stick skeleton (ground position + person-sized height) per
image, turns it into a bbox mask, and runs the trained edit adapter to fill it.
Background outside the mask is preserved 100% (hard-restore).

The skeleton decides WHERE/scale (not exact pose). 1–N people per image.
No training; loads the pipeline once. Needs GPU + SD3.5 access + the trained
adapter (mount your run as a dataset).

## 1. Setup

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import transformers, diffusers, torch
assert transformers.__version__ == '4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
assert torch.cuda.is_available(); print('OK on', torch.cuda.get_device_name(0))

## 2. SD3.5 access + locate trained adapter

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'Need HF_TOKEN or mounted SD3.5'
    from huggingface_hub import login; login(token=HF_TOKEN)

ADAPTER_DIR = None   # set explicitly, else auto-find
def _autofind():
    c = list(Path('/kaggle/input').rglob('input_adapter.pt'))
    c += list(Path('/kaggle/working/vin_lora/models').rglob('input_adapter.pt'))
    return c[0].parent if c else None
adir = Path(ADAPTER_DIR) if ADAPTER_DIR else _autofind()
assert adir and (adir/'pytorch_lora_weights.safetensors').exists() and (adir/'input_adapter.pt').exists(), \
    'Adapter not found — mount run as dataset (adapter/ with both files) or set ADAPTER_DIR'
print('adapter:', adir)

## 3. Load the edit runner (once)

In [ ]:
from LoRA.inference.sd35_edit_runner import SD35EditRunner
runner = SD35EditRunner(SD35_MODEL, adir, hf_token=HF_TOKEN).load()
print('runner ready')

## 4. Background images + random-skeleton settings

Put background images (no people, or any scene) under `BG_DIR`. Tune placement
in `PlacementConfig` (height range, how many people, feet band).

In [ ]:
from pathlib import Path
from LoRA.inference.random_skeleton import PlacementConfig
BG_DIR = Path('/kaggle/working/my_backgrounds')   # <- put your background images here
BG_DIR.mkdir(parents=True, exist_ok=True)
bg_paths = sorted([p for p in BG_DIR.glob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg')])
assert bg_paths, f'No images in {BG_DIR} — upload background images there.'

CFG = PlacementConfig(min_height_frac=0.25, max_height_frac=0.55,
                      min_people=1, max_people=3)
SEED = 42; STEPS = 30
PROMPT = 'a photo of <vin_ped> pedestrian, a person, natural lighting'
print(len(bg_paths), 'background images')

## 5. Random skeleton -> mask -> edit

In [ ]:
from PIL import Image
from LoRA.inference.random_skeleton import random_placement, draw_skeleton_overlay
OUT = Path('/kaggle/working/skeleton_edit_outputs'); (OUT/'images').mkdir(parents=True, exist_ok=True)
runner.precompute_embeds([PROMPT])
results = []
for i, bp in enumerate(bg_paths):
    bg = Image.open(bp).convert('RGB')
    mask, bboxes, sk = random_placement(bg.size, seed=SEED, index=i, cfg=CFG)
    overlay = draw_skeleton_overlay(bg, bboxes, sk)
    out = runner.edit(bg, mask, PROMPT, seed=SEED, num_inference_steps=STEPS)
    out.save(OUT/'images'/f'{bp.stem}_edit.png')
    results.append((bg, mask, overlay, out, bp.stem))
    print('done', bp.name, '->', len(bboxes), 'people')
print('outputs ->', OUT)

## 6. Show: background | skeleton overlay | mask | edit

In [ ]:
from PIL import Image
from IPython.display import display
for bg, mask, overlay, out, name in results:
    cells = [bg, overlay, mask.convert('RGB'), out]
    cells = [im.resize((256,256)) for im in cells]
    strip = Image.new('RGB',(256*4,256))
    for j,im in enumerate(cells): strip.paste(im,(256*j,0))
    print(name, '(bg | skeleton | mask | edit)'); display(strip)